In [ ]:
# ── Pretrained checkpoint ─────────────────────────────────────────────────────
CORRELATION         = 0.00
PRETRAIN_CHECKPOINT = f"pretrain_corr_{CORRELATION:.2f}.pth"

# ── Finetuning hyperparameters ────────────────────────────────────────────────
CONCENTRATION         = .9      # weight given to count_a; rest split evenly
FINETUNE_STEPS        = 30_00
FINETUNE_BATCH_SIZE   = 96
FINETUNE_LR           = 1e-5
FINETUNE_MIN_LR       = 1e-7
FINETUNE_WARMUP_RATIO = 0.2
MAX_GRAD_NORM         = 1.0

# ── Logging & evaluation ──────────────────────────────────────────────────────
LOG_INTERVAL = 100   # steps between each log + eval

# Standard metrics — any subset of ["loss", "answer_acc"]
METRICS = ["loss", "answer_acc"]

# ── Gradient / drift metric settings ─────────────────────────────────────────
# Batch size for per-task gradient probes (can be < FINETUNE_BATCH_SIZE)
GRAD_PROBE_BATCH_SIZE = 32

# ── Tasks ─────────────────────────────────────────────────────────────────────
FINETUNE_TASK_NAMES = [
    "count_a",                                          # special task — first
    "count_b", "count_c",
    "count_aa", "count_bb", "count_cc",
    "index_a",  "index_b",  "index_c",
    "index_aa", "index_bb", "index_cc",
    "token_at_40",
]
SPECIAL_TASK = "count_a"   # gradient of this task is compared against all others

# ── Eval dataset sizes ────────────────────────────────────────────────────────
VAL_EXAMPLES        = 500
EVAL_PER_OTHER_TASK = 500

# ── Model architecture (must match checkpoint) ────────────────────────────────
MODEL_CFG = dict(
    block_size  = 512,
    n_layer     = 6,
    n_head      = 6,
    n_embd      = 192,
    embd_pdrop  = 0.1,
    resid_pdrop = 0.1,
    attn_pdrop  = 0.1,
)

# ── PCFG pools ────────────────────────────────────────────────────────────────
POOL_N_CORRELATED   = 100_00
POOL_N_UNCORRELATED = 100_00
CHUNK_SIZE          = 250

# ── Optimizer ─────────────────────────────────────────────────────────────────
OPTIMIZER_CFG = dict(type="AdamW", weight_decay=0.0, betas=[0.9, 0.95], eps=1e-8)

# ── Misc ──────────────────────────────────────────────────────────────────────
SEED   = 42
DEVICE = "auto"   # "auto" | "cpu" | "cuda:0"

import os, math, random, itertools
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

from config       import CFG
from config_utils import (
    build_optimizer, build_task_registry,
    get_device, get_warmup_steps, set_seed,
)
from mingpt   import GPT, GPTConfig
from pcfg_gen import (
    CharTokenizer, PCFGDataset, PCFGGenerator,
    generate_dataset, build_pools, collate_fn,
)
from train_help import (
    sample_batch, get_cosine_lr,
    _build_val_loaders, _evaluate_loader,
    calculate_answer_accuracy,
)

set_seed(SEED)
device = get_device(DEVICE)
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

pcfg = PCFGGenerator()

pools = build_pools(
    pcfg_gen       = pcfg,
    n_correlated   = POOL_N_CORRELATED,
    n_uncorrelated = POOL_N_UNCORRELATED,
    chunk_size     = CHUNK_SIZE,
    verbose        = True,
)
print(f"\nPool sizes — correlated: {len(pools['correlated']):,} | "
      f"uncorrelated: {len(pools['uncorrelated']):,}")

tokenizer     = CharTokenizer()
task_registry = build_task_registry(CFG["task_definitions"])

ALL_TASKS   = CFG["task_sets"]["all"]
OTHER_TASKS = [t for t in ALL_TASKS if t not in ("count_a", "count_b")]

MAX_LEN          = CFG["tokenizer"]["max_length"]
MASK_ANSWER_ONLY = CFG["tokenizer"]["mask_answer_only_val"]

print("Generating count_a eval set …")
count_a_examples = generate_dataset(
    n_examples=VAL_EXAMPLES, task_names=["count_a"],
    pcfg_gen=pcfg, task_reg=task_registry, chunk_size=CHUNK_SIZE,
)

print("Generating count_b eval set …")
count_b_examples = generate_dataset(
    n_examples=VAL_EXAMPLES, task_names=["count_b"],
    pcfg_gen=pcfg, task_reg=task_registry, chunk_size=CHUNK_SIZE,
)

print(f"Generating other-task eval sets ({len(OTHER_TASKS)} tasks × {EVAL_PER_OTHER_TASK}) …")
other_examples = []
for t in OTHER_TASKS:
    other_examples += generate_dataset(
        n_examples=EVAL_PER_OTHER_TASK, task_names=[t],
        pcfg_gen=pcfg, task_reg=task_registry, chunk_size=CHUNK_SIZE,
    )

eval_datasets = {
    "count_a": PCFGDataset(
        count_a_examples, tokenizer, max_length=MAX_LEN, mask_answer_only=MASK_ANSWER_ONLY),
    "count_b": PCFGDataset(
        count_b_examples, tokenizer, max_length=MAX_LEN, mask_answer_only=MASK_ANSWER_ONLY),
    "all_other_avg": PCFGDataset(
        other_examples,   tokenizer, max_length=MAX_LEN, mask_answer_only=MASK_ANSWER_ONLY),
}

print("\nEval dataset sizes:")
for name, ds in eval_datasets.items():
    print(f"  {name:<18} {len(ds):>6} examples")

gpt_config = GPTConfig(
    vocab_size  = tokenizer.vocab_size,
    block_size  = MODEL_CFG["block_size"],
    n_layer     = MODEL_CFG["n_layer"],
    n_head      = MODEL_CFG["n_head"],
    n_embd      = MODEL_CFG["n_embd"],
    embd_pdrop  = MODEL_CFG["embd_pdrop"],
    resid_pdrop = MODEL_CFG["resid_pdrop"],
    attn_pdrop  = MODEL_CFG["attn_pdrop"],
)

model = GPT(gpt_config).to(device)

assert os.path.exists(PRETRAIN_CHECKPOINT), f"Checkpoint not found: {PRETRAIN_CHECKPOINT}"
checkpoint = torch.load(PRETRAIN_CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print(f"Loaded  : {PRETRAIN_CHECKPOINT}")
print(f"Corr    : {checkpoint.get('correlation', 'unknown')}")

# ── Auto-detect one prefix per transformer block ──────────────────────────────
layer_prefixes = sorted({
    ".".join(n.split(".")[:3])          # "transformer.h.0", "transformer.h.1", …
    for n, _ in model.named_parameters()
    if n.startswith("transformer.h.")
})
print(f"Tracking {len(layer_prefixes)} layers: {layer_prefixes}")


# ── Snapshot of weights at the last eval step (for incremental drift) ────────
# Updated after every evaluation; first interval measures from the initial
# pretrained weights loaded above.
last_eval_params = {
    name: param.detach().cpu().float().clone()
    for name, param in model.named_parameters()
}


def get_layer_grads(batch):
    """
    Forward + backward for one batch. Returns a dict:
        { layer_prefix: flat_grad_vector (cpu float32) }
    No optimizer step; gradients are zeroed afterwards.
    """
    model.train()
    model.zero_grad()
    _, loss = model(batch["input_ids"].to(device), batch["target_ids"].to(device))
    loss.backward()

    grads = {}
    for prefix in layer_prefixes:
        vecs = [
            p.grad.detach().cpu().float().flatten()
            for n, p in model.named_parameters()
            if n.startswith(prefix) and p.grad is not None
        ]
        if vecs:
            grads[prefix] = torch.cat(vecs)

    model.zero_grad()
    return grads


def cossim(a, b):
    """Cosine similarity between two 1-D tensors."""
    return (a @ b / (a.norm() * b.norm()).clamp(min=1e-12)).item()


def probe_batch(task_name):
    """Sample a small single-task batch for gradient probing."""
    return sample_batch(
        batch_size       = GRAD_PROBE_BATCH_SIZE,
        task_names       = [task_name],
        task_weights     = [1.0],
        pcfg_gen         = pcfg,
        task_reg         = task_registry,
        tokenizer        = tokenizer,
        chunk_size       = CHUNK_SIZE,
        mask_answer_only = False,
        data_pools       = pools,
        correlation      = CORRELATION,
    )


def compute_grad_and_drift_metrics():
    """
    Compute all three layerwise metrics at the current model weights.

    Returns
    -------
    cossim_sv_o  : list[float]  len = n_layers
        Cosine sim between special-task gradient and mean-other gradient.

    cossim_o_o   : list[float]  len = n_layers
        Mean pairwise cosine sim among all other-task gradients.

    drift        : list[float]  len = n_layers
        L2 distance of current weights from base weights.
    """
    other_tasks = [t for t in FINETUNE_TASK_NAMES if t != SPECIAL_TASK]

    # Gradient vectors per task per layer
    special_grads = get_layer_grads(probe_batch(SPECIAL_TASK))
    other_grads   = {t: get_layer_grads(probe_batch(t)) for t in other_tasks}

    cossim_sv_o, cossim_o_o, drift = [], [], []

    for prefix in layer_prefixes:

        # ── special vs. mean-other ────────────────────────────────────────────
        g_sp = special_grads.get(prefix)
        other_vecs = [other_grads[t][prefix] for t in other_tasks if prefix in other_grads[t]]

        if g_sp is not None and other_vecs:
            g_other_mean = torch.stack(other_vecs).mean(0)
            cossim_sv_o.append(cossim(g_sp, g_other_mean))
        else:
            cossim_sv_o.append(float("nan"))

        # ── mean pairwise among other tasks ───────────────────────────────────
        if len(other_vecs) >= 2:
            pairs = list(itertools.combinations(other_vecs, 2))
            cossim_o_o.append(np.mean([cossim(a, b) for a, b in pairs]))
        elif len(other_vecs) == 1:
            cossim_o_o.append(1.0)
        else:
            cossim_o_o.append(float("nan"))

        # ── weight drift from last eval step ────────────────────────────────────
        diffs = [
            (p.detach().cpu().float() - last_eval_params[n]).flatten()
            for n, p in model.named_parameters()
            if n.startswith(prefix) and n in last_eval_params
        ]
        drift.append(torch.cat(diffs).norm().item() if diffs else float("nan"))

    return cossim_sv_o, cossim_o_o, drift


In [ ]:

# ── Task weights ──────────────────────────────────────────────────────────────
n_other          = len(FINETUNE_TASK_NAMES) - 1
other_w          = (1.0 - CONCENTRATION) / n_other
finetune_weights = [CONCENTRATION] + [other_w] * n_other

print("Task weights:")
for name, w in zip(FINETUNE_TASK_NAMES, finetune_weights):
    print(f"  {name:<18} {w:.4f}  {'█' * int(round(w * 40))}")

# ── Optimizer & schedule ──────────────────────────────────────────────────────
optimizer    = build_optimizer(model.parameters(), OPTIMIZER_CFG, FINETUNE_LR)
warmup_steps = get_warmup_steps(FINETUNE_STEPS, warmup_ratio=FINETUNE_WARMUP_RATIO)
val_loaders  = _build_val_loaders(eval_datasets, FINETUNE_BATCH_SIZE, tokenizer)
metrics_set  = set(METRICS)

# ── History dict ──────────────────────────────────────────────────────────────
history = {
    "steps"            : [],
    "train_loss"       : [],
    "train_answer_acc" : [],
    "val"              : {name: {"loss": [], "answer_acc": []} for name in val_loaders},
    # ── new metrics (one row per eval step, one value per layer) ──────────────
    # grad_cossim_special_vs_other[step][layer]
    #   → cosine sim between count_a gradient and mean gradient of all other tasks
    "grad_cossim_special_vs_other" : [],
    # grad_cossim_other_vs_other[step][layer]
    #   → mean pairwise cosine sim among gradients of every non-count_a task
    "grad_cossim_other_vs_other"   : [],
    # weight_drift[step][layer]
    #   → L2 distance of current layer weights from the frozen pretrained weights
    "weight_drift"                 : [],
}

print(f"\nFinetuning for {FINETUNE_STEPS:,} steps  |  "
      f"LR {FINETUNE_LR} → {FINETUNE_MIN_LR}  |  "
      f"eval every {LOG_INTERVAL} steps\n" + "=" * 80)

# ── Training loop ─────────────────────────────────────────────────────────────
for step in range(1, FINETUNE_STEPS + 1):

    # LR schedule
    cur_lr = get_cosine_lr(step, FINETUNE_STEPS, FINETUNE_LR, FINETUNE_MIN_LR, warmup_steps)
    for pg in optimizer.param_groups:
        pg["lr"] = cur_lr

    # Sample mixed-task batch and take a gradient step
    batch = sample_batch(
        batch_size       = FINETUNE_BATCH_SIZE,
        task_names       = FINETUNE_TASK_NAMES,
        task_weights     = finetune_weights,
        pcfg_gen         = pcfg,
        task_reg         = task_registry,
        tokenizer        = tokenizer,
        chunk_size       = CHUNK_SIZE,
        mask_answer_only = False,
        data_pools       = pools,
        correlation      = CORRELATION,
    )

    model.train()
    optimizer.zero_grad()
    logits, loss = model(batch["input_ids"].to(device), batch["target_ids"].to(device))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()

    train_acc, _ = calculate_answer_accuracy(logits, batch["target_ids"].to(device),
                                             batch["answer_positions"])

    # ── Eval at every LOG_INTERVAL ────────────────────────────────────────────
    if step % LOG_INTERVAL == 0:
        history["steps"].append(step)
        history["train_loss"].append(loss.item())
        history["train_answer_acc"].append(train_acc)

        log_parts = [
            f"step {step}/{FINETUNE_STEPS}",
            f"loss={loss.item():.4f}",
            f"acc={train_acc:.3f}",
            f"lr={cur_lr:.2e}",
        ]

        # Standard val metrics
        for name, loader in val_loaders.items():
            v_loss, v_acc = _evaluate_loader(model, loader, device, metrics_set)
            history["val"][name]["loss"].append(v_loss)
            history["val"][name]["answer_acc"].append(v_acc)
            log_parts += [f"{name}_loss={v_loss:.4f}", f"{name}_acc={v_acc:.3f}"]

        # Gradient cosine-sim + weight-drift metrics
        sv_o, o_o, drift = compute_grad_and_drift_metrics()
        history["grad_cossim_special_vs_other"].append(sv_o)
        history["grad_cossim_other_vs_other"].append(o_o)
        history["weight_drift"].append(drift)

        # Advance the snapshot so the next interval is measured from here
        last_eval_params = {
            name: param.detach().cpu().float().clone()
            for name, param in model.named_parameters()
        }

        log_parts += [
            f"grad_sim(spc↔other)={np.nanmean(sv_o):.3f}",
            f"grad_sim(other↔other)={np.nanmean(o_o):.3f}",
            f"drift={np.nanmean(drift):.4f}",
        ]

        print(" | ".join(log_parts))

print("\nDone.")

SPLIT_COLORS = {
    "count_a"      : "#e63946",
    "count_b"      : "#457b9d",
    "all_other_avg": "#2a9d8f",
}
TRAIN_COLOR   = "#f4a261"
METRIC_LABELS = {"loss": "Cross-entropy loss", "answer_acc": "Answer accuracy"}

steps = history["steps"]

fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 4), squeeze=False)

for col, metric in enumerate(METRICS):
    ax = axes[0][col]

    # Training curve
    train_vals = history.get(f"train_{metric}", [])
    if train_vals:
        ax.plot(steps, train_vals, color=TRAIN_COLOR, lw=1.5,
                ls="--", label="train", alpha=0.8)

    # Validation curves
    for split_name, split_data in history["val"].items():
        vals = split_data.get(metric, [])
        if vals:
            ax.plot(steps, vals, color=SPLIT_COLORS.get(split_name),
                    lw=2, label=f"val – {split_name}")

    ax.set_title(METRIC_LABELS.get(metric, metric))
    ax.set_xlabel("Step")
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Finetuning  |  corr={CORRELATION:.2f}  conc={CONCENTRATION:.2f}  "
    f"steps={FINETUNE_STEPS:,}",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

# Convert to numpy arrays: shape (n_eval_steps, n_layers)
sv_o_arr = np.array(history["grad_cossim_special_vs_other"])   # special vs other
o_o_arr  = np.array(history["grad_cossim_other_vs_other"])     # other  vs other
n_layers = len(layer_prefixes)
layer_labels = [f"L{i}" for i in range(n_layers)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, arr, title in [
    (axes[0], sv_o_arr, f"Grad cosine sim: {SPECIAL_TASK} ↔ mean(other tasks)"),
    (axes[1], o_o_arr,  "Grad cosine sim: mean pairwise among other tasks"),
]:
    for i in range(n_layers):
        ax.plot(steps, arr[:, i], label=layer_labels[i], lw=1.5)
    ax.axhline(0, color="black", lw=0.7, ls="--")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel("Cosine similarity")
    ax.legend(fontsize=7, ncol=2)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

drift_arr = np.array(history["weight_drift"])   # (n_eval_steps, n_layers)

fig, ax = plt.subplots(figsize=(8, 4))
for i in range(n_layers):
    ax.plot(steps, drift_arr[:, i], label=layer_labels[i], lw=1.5)

ax.set_title("Weight drift since last eval step (L2 norm per layer)")
ax.set_xlabel("Step")
ax.set_ylabel("‖θ − θ_prev‖₂")
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(24, 3))

# ── Panel 1: accuracy over time ───────────────────────────────────────────────
ax = axes[0]
acc_series = {
    "count_a"      : history["val"]["count_a"]["answer_acc"],
    "count_b"      : history["val"]["count_b"]["answer_acc"],
    "all_other_avg": history["val"]["all_other_avg"]["answer_acc"],
}
acc_colors = {
    "count_a"      : "#e63946",
    "count_b"      : "#457b9d",
    "all_other_avg": "#2a9d8f",
}
for split_name, vals in acc_series.items():
    ax.plot(steps, vals, color=acc_colors[split_name], lw=2, label=split_name)
ax.set_title("Answer accuracy per task group", fontsize=9)
ax.set_xlabel("Step")
ax.set_ylabel("Accuracy")
ax.legend(fontsize=7)
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(True, alpha=0.3)

# ── Panels 2-4: heatmaps ──────────────────────────────────────────────────────
data_panels = [
    (sv_o_arr.T,  f"Grad cosine sim\n{SPECIAL_TASK} ↔ mean(other)", "RdBu", (-1, 1)),
    (o_o_arr.T,   "Grad cosine sim\nmean pairwise other↔other",       "RdBu", (-1, 1)),
    (drift_arr.T, "Weight drift\n‖θ − θ_prev‖₂",                     "Reds", (0, None)),
]
for ax, (data, title, cmap, (vmin, vmax)) in zip(axes[1:], data_panels):
    kwargs = dict(aspect="auto", origin="lower", cmap=cmap)
    if vmin is not None: kwargs["vmin"] = vmin
    if vmax is not None: kwargs["vmax"] = vmax
    im = ax.imshow(data, **kwargs,
                   extent=[steps[0], steps[-1], -0.5, n_layers - 0.5])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Step")
    ax.set_ylabel("Layer")
    ax.set_yticks(range(n_layers))
    ax.set_yticklabels(layer_labels, fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.85)

plt.suptitle(
    f"corr={CORRELATION:.2f}  conc={CONCENTRATION:.2f}",
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.show()

# ── Final-step summary table ──────────────────────────────────────────────────
print(f"{'Split':<20}" + "".join(f"{m:>16}" for m in METRICS))
print("-" * (20 + 16 * len(METRICS)))
for split_name, split_data in history["val"].items():
    row = f"{split_name:<20}"
    for m in METRICS:
        vals = split_data.get(m, [])
        row += f"{vals[-1]:>16.4f}" if vals else f"{'nan':>16}"
    print(row)

print()
print(f"{'Layer':<16}  grad_sim(spc↔other)  grad_sim(other↔other)  weight_drift")
print("-" * 72)
for i, prefix in enumerate(layer_prefixes):
    sv = sv_o_arr[-1, i]
    oo = o_o_arr[-1, i]
    dr = drift_arr[-1, i]
    print(f"  {prefix:<14}  {sv:>19.4f}  {oo:>21.4f}  {dr:>12.4f}")

# # ── Save checkpoint ───────────────────────────────────────────────────────────
# save_path = f"finetune_corr_{CORRELATION:.2f}_conc_{CONCENTRATION:.2f}.pth"
# torch.save({
#     "model_state_dict"    : model.state_dict(),
#     "optimizer_state_dict": optimizer.state_dict(),
#     "correlation"         : CORRELATION,
#     "concentration"       : CONCENTRATION,
#     "history"             : history,
#     "layer_prefixes"      : layer_prefixes,
#     "stage"               : "finetune",
# }, save_path)
# print(f"\nSaved → {save_path}")





